In [1]:
#create CSV output from ESM, can be used in main branch to create CSV output of capacity and capacity addition

from zen_garden.postprocess.results import Results
import pandas as pd
from pathlib import Path

# ---------------- Configuration ----------------
#change according to your setup
dataset_name = "energy_transition_example"
base_path = Path(r".\outputs")
results_path = base_path / dataset_name
output_dir = Path("./CSV_output") / dataset_name
output_dir.mkdir(parents=True, exist_ok=True)

# ---------------- Load Results ----------------
res_basic = Results(results_path)

capacity = res_basic.get_full_ts("capacity").reset_index()
capacity_addition = res_basic.get_full_ts("capacity_addition").reset_index()
capacity_previous = res_basic.get_full_ts("capacity_previous").reset_index()

# ---------------- Filter Technologies ----------------
selected_technologies = {"heat_pump", "photovoltaics", "wind_offshore", "wind_onshore"}

capacity = capacity[capacity["technology"].isin(selected_technologies)]
capacity_addition = capacity_addition[capacity_addition["technology"].isin(selected_technologies)]
capacity_previous = capacity_previous[capacity_previous["technology"].isin(selected_technologies)]

# ---------------- Adjust capacity_addition["0"] by subtracting capacity_previous["0"] ----------------
# Ensure all year columns are strings
capacity_addition.columns = capacity_addition.columns.map(str)
capacity_previous.columns = capacity_previous.columns.map(str)

# Identify the first year column (usually '0')
year_cols = [col for col in capacity_addition.columns if col.isdigit()]
first_year_col = min(year_cols, key=int)

# Prepare capacity_previous for merging
cap_prev_trimmed = capacity_previous[["technology", "capacity_type", "location", first_year_col]].copy()
cap_prev_trimmed = cap_prev_trimmed.rename(columns={first_year_col: "previous_capacity"})

# Merge and subtract
capacity_addition = pd.merge(
    capacity_addition,
    cap_prev_trimmed,
    on=["technology", "capacity_type", "location"],
    how="left"
)
capacity_addition[first_year_col] = capacity_addition[first_year_col] - capacity_addition["previous_capacity"].fillna(0)
capacity_addition[first_year_col] = capacity_addition[first_year_col].clip(lower=0)
capacity_addition = capacity_addition.drop(columns=["previous_capacity"])

# ---------------- Save Filtered CSVs ----------------
# and round to 4 decimal digits
capacity.round(4).to_csv(output_dir / "capacity.csv", index=False)
capacity_addition.round(4).to_csv(output_dir / "capacity_addition.csv", index=False)
capacity_previous.round(4).to_csv(output_dir / "capacity_previous.csv", index=False)

# ---------------- Aggregation Setup ----------------
specific_locations = {"DE", "CH", "IT", "CZ", "SE", "UK", "DK", "NL", "AT"}
all_locations = set(capacity["location"].unique())
roe_locations = all_locations - specific_locations


def aggregate_by_location(df, location_set, location_name):
    return (
        df[df["location"].isin(location_set)]
        .groupby(["technology", "capacity_type"])
        .sum(numeric_only=True)
        .assign(location=location_name)
    )


# ---------------- Aggregate Data ----------------
capacity_aggregated = pd.concat([
    aggregate_by_location(capacity, {"DE"}, "DE"),
    aggregate_by_location(capacity, {"CH"}, "CH"),
    aggregate_by_location(capacity, {"DK"}, "DK"),
    aggregate_by_location(capacity, {"NL"}, "NL"),
    aggregate_by_location(capacity, {"SE"}, "SE"),
    aggregate_by_location(capacity, {"UK"}, "UK"),
    aggregate_by_location(capacity, {"IT"}, "IT"),
    aggregate_by_location(capacity, {"CZ"}, "CZ"),
    aggregate_by_location(capacity, {"AT"}, "AT"),

    aggregate_by_location(capacity, roe_locations, "ROE")
]).reset_index()

capacity_addition_aggregated = pd.concat([
    aggregate_by_location(capacity_addition, {"DE"}, "DE"),
    aggregate_by_location(capacity_addition, {"CH"}, "CH"),
    aggregate_by_location(capacity_addition, {"DK"}, "DK"),
    aggregate_by_location(capacity_addition, {"NL"}, "NL"),
    aggregate_by_location(capacity_addition, {"SE"}, "SE"),
    aggregate_by_location(capacity_addition, {"UK"}, "UK"),
    aggregate_by_location(capacity_addition, {"IT"}, "IT"),
    aggregate_by_location(capacity_addition, {"CZ"}, "CZ"),
    aggregate_by_location(capacity_addition, {"AT"}, "AT"),

    aggregate_by_location(capacity_addition, roe_locations, "ROE")
]).reset_index()

# ---------------- Save Aggregated CSVs ----------------
capacity_aggregated.round(4).to_csv(output_dir / "capacity_aggregated_by_location.csv", index=False)
capacity_addition_aggregated.round(4).to_csv(output_dir / "capacity_addition_aggregated_by_location.csv", index=False)

print(f"✅ All CSV files for dataset '{dataset_name}' saved successfully in: {output_dir}")

✅ All CSV files for dataset 'energy_transition_example' saved successfully in: CSV_output\energy_transition_example


In [2]:
# --- Update aggregated CSV column headers with actual years from system.json ---
import json
# Load system.json
system_path = base_path.parent / dataset_name / "system.json"
with open(system_path, "r") as f:
    system_config = json.load(f)

reference_year = system_config["reference_year"]
interval = system_config["interval_between_years"]
optimized_years = system_config["optimized_years"]
year_labels = [str(reference_year + i * interval) for i in range(optimized_years)]

# Create rename mapping from '0', '1', ... to actual year labels
rename_columns = {str(i): year_labels[i] for i in range(len(year_labels))}

# Function to rename year columns in a CSV
def rename_year_columns(csv_path):
    df = pd.read_csv(csv_path)
    df = df.rename(columns=rename_columns)
    df.to_csv(csv_path, index=False)
    print(f"Renamed year columns in: {csv_path}")

# Apply to both aggregated files
rename_year_columns(output_dir / "capacity_aggregated_by_location.csv")
rename_year_columns(output_dir / "capacity_addition_aggregated_by_location.csv")


Renamed year columns in: CSV_output\energy_transition_example\capacity_aggregated_by_location.csv
Renamed year columns in: CSV_output\energy_transition_example\capacity_addition_aggregated_by_location.csv
